# aw_05_b1b2 — Stages B1/B2: Phase-1 general SFT, two arms (Track B, RQ1+RQ2)

**Protocol**: §5.1 B1/B2. Two arms differing ONLY in response provenance:
- **B1**: curated corpus (`data/p1/p1_general_sft.jsonl`)
- **B2**: rejection-sampled corpus — base-model samples gated by ExactAnswerVerifier
  (`scripts/build_p1_rs_data.py`), same prompt set.

Readouts per arm: (1) P1 general held-out accuracy (`run_p1_eval.py`, retention anchor
is the BASE model's score), (2) **transfer probe** — frozen 200-step PlayWorld SFT
(`probe_playworld_sft.yaml`) evaluated on the frozen suites; probe eval-ID goal-valid
accuracy is the Phase-1 champion primary metric (§6).

**Repos**: B1 → `m97j/aw-runs-b1`, B2 → `m97j/aw-runs-b2` (one training run per repo;
probe runs live in the SAME repo as their parent under a second run — acceptable since
fetch_run pins by run_id? NO — fetch_run reads repo-root artifacts. Probe runs get their
own repos `m97j/aw-runs-b1-probe`, `m97j/aw-runs-b2-probe`.)


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title a_b12_data — P1 mixture + held-out + RS corpus
!python scripts/build_p1_data.py
# base-model reference on the held-out (retention anchor, run once)
!python scripts/run_p1_eval.py \
  --config configs/experiments/b1_general_sft.yaml \
  --holdout data/p1/p1_general_holdout.jsonl \
  --label qwen3-8b-base --output runs/p1_eval_base.json \
  --hf-sync-repo m97j/aw-runs-b1
# B2 rejection-sampled corpus (generation-heavy)
!python scripts/build_p1_rs_data.py \
  --config configs/experiments/b1_general_sft.yaml \
  --input data/p1/p1_general_sft.jsonl \
  --output data/p1/p1_general_sft_rs.jsonl \
  --num-candidates 4 --batch-size 64 \
  --hf-sync-repo m97j/axiom-general-posttrain


In [ ]:
# @title b_b1_train — P1 general SFT (curated arm)
!python scripts/run_experiment.py \
  --config configs/experiments/b1_general_sft.yaml \
  --hf-sync-repo m97j/aw-runs-b1


In [ ]:
# @title b_b2_train — P1 general SFT (rejection-sampled arm)
!python scripts/run_experiment.py \
  --config configs/experiments/b2_general_sft_rs.yaml \
  --hf-sync-repo m97j/aw-runs-b2


In [ ]:
# @title c_b12_p1_eval — held-out accuracy for both arms (retention readout)
B1_RUN_ID = ""  # <- from b_b1_train "run_id: ..."
B2_RUN_ID = ""  # <- from b_b2_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1 --run-id {B1_RUN_ID}
b1_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2 --run-id {B2_RUN_ID}
b2_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_p1_eval.py \
  --config configs/experiments/b1_general_sft.yaml \
  --adapter-dir {b1_dir} --label b1-sft \
  --output runs/p1_eval_b1.json --hf-sync-repo m97j/aw-runs-b1
!python scripts/run_p1_eval.py \
  --config configs/experiments/b2_general_sft_rs.yaml \
  --adapter-dir {b2_dir} --label b2-sft-rs \
  --output runs/p1_eval_b2.json --hf-sync-repo m97j/aw-runs-b2


In [ ]:
# @title d_b12_probe — frozen transfer probe (200 steps) per arm
!python scripts/build_training_data.py
!python scripts/build_eval_suites.py --episodes-per-suite 300

import json
from pathlib import Path


def get_adapter_sha(run_id: str) -> str:
    """Safely extracts the SHA from the run lineage artifact."""
    path = Path("runs") / run_id / "artifacts" / "lineage.json"
    data = json.loads(path.read_text())
    return data["output_adapter_sha256"]

b1_sha = get_adapter_sha(B1_RUN_ID)
b2_sha = get_adapter_sha(B2_RUN_ID)

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b1_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b1 \
  --override lineage.parent_adapter.sha256={b1_sha} \
  --override experiment_name=probe-playworld-sft-b1 \
  --hf-sync-repo m97j/aw-runs-b1-probe

!python scripts/run_experiment.py \
  --config configs/experiments/probe_playworld_sft.yaml \
  --parent-adapter-dir {b2_dir} \
  --override lineage.parent_adapter.repo_id=m97j/aw-runs-b2 \
  --override lineage.parent_adapter.sha256={b2_sha} \
  --override experiment_name=probe-playworld-sft-b2 \
  --hf-sync-repo m97j/aw-runs-b2-probe


In [ ]:
# @title e_b12_probe_eval — probe adapters on the frozen suites
B1_PROBE_RUN = ""  # <- from d_b12_probe outputs
B2_PROBE_RUN = ""

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1_PROBE_RUN}
b1p_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
out = !python scripts/fetch_run.py --repo m97j/aw-runs-b2-probe --run-id {B2_PROBE_RUN}
b2p_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b1p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b1-probe
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b2p_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b2-probe


In [ ]:
# @title f_b12_analysis — probe-vs-probe and probe-vs-A1
B1_PROBE_EVAL = ""  # <- eval run ids from e_b12_probe_eval
B2_PROBE_EVAL = ""
A1_EVAL = "20260801-063425--eval-playworld--s42--3bf440"

!python scripts/fetch_run.py --repo m97j/aw-runs-b1-probe --run-id {B1_PROBE_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b2-probe --run-id {B2_PROBE_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B1_PROBE_EVAL} --label-a b1-probe \
  --run-b runs/{B2_PROBE_EVAL} --label-b b2-probe \
  --output runs/{B1_PROBE_EVAL}/analysis_b1_vs_b2_probe.json --hf-sync-repo m97j/aw-runs-b1-probe

!python scripts/run_analysis.py \
  --run-a runs/{B1_PROBE_EVAL} --label-a b1-probe \
  --run-b runs/{A1_EVAL} --label-b a1-sft \
  --output runs/{B1_PROBE_EVAL}/analysis_b1probe_vs_a1.json --hf-sync-repo m97j/aw-runs-b1-probe


## Stage checklist (feeds §6 Phase-1 champion selection)
- [ ] base / B1 / B2 held-out accuracies recorded (retention: drop ≤ 3 pts vs base)
- [ ] RS manifest: acceptance_rate + coverage reported
- [ ] probe eval-ID goal-valid accuracy per arm = Phase-1 primary metric input
- [ ] Winner(s) proceed to B3 (P1 DPO, aw_06)
